# Stenosis Detection — Kaggle BUILD notebook

**YOLO11s @ 768px on ARCADE task-2 + Danilov.** No nnU-Net teacher (detection = one YOLO train).
ARCADE-only/nano/640 sat at F1 0.25 — this recipe (bigger model, higher res, more data, SSL) targets the **F1 > 0.57** floor.

### Kaggle setup (do this first)
1. **Settings → Accelerator → GPU** (T4 x2 or P100).
2. **Settings → Internet → ON** (pip + git clone + Danilov download).
3. **Add Input → Datasets**: attach your **ARCADE** dataset. Danilov is auto-downloaded below (no upload needed).
4. **Run All** (`DRY_RUN = True` first for a wiring check, then `False`).

Each run auto-names its folder via `run_tag(cfg)` (e.g. `arcade+danilov_yolo11s_768_e150`), so re-runs don't clobber.
All heavy logic is in the importable `src/*` package; this notebook just orchestrates it.

## 1 · GPU + code + install

In [ ]:
import os, sys, torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU.'
GITHUB_TOKEN = ''                                             # '' if repo PUBLIC, else paste a PAT
REPO_SLUG = 'jugalmodi0111/interventional-imaging-pipeline'
REPO = '/kaggle/working/interventional-imaging-pipeline'
if not os.path.exists(REPO):
    auth = f'{GITHUB_TOKEN}@' if GITHUB_TOKEN else ''
    !git clone https://{auth}github.com/{REPO_SLUG}.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install 'ultralytics>=8.2' pycocotools opencv-python pyyaml
# optional — only if you set cfg['ssl']['seed'] = 'gdino' below:
# !pip -q install transformers
from src import env
E = env.setup()                                              # platform=kaggle, paths under /kaggle/working
torch.backends.cudnn.benchmark = True                        # speed: cuDNN autotune for the fixed 640x640 input (quality-neutral)

## 2 · Point the config at your attached datasets
Kaggle mounts datasets read-only under `/kaggle/input/`. Find the exact folder, then symlink it into `data/raw/` so the configs resolve unchanged. `ARCADE_INPUT` must be the folder that **contains** `stenosis/` (and `syntax/`).

## 2a · Attach Danilov as a Kaggle Dataset (upload, same as ARCADE)
**Do not download it inside the notebook** — the 7.7 GB zip extracts to ~25 GB of BMPs and overflows `/kaggle/working` (~20 GB) with `write error (disk full?)`.

One-time setup (exactly how you attached ARCADE):
1. On your PC, download the zip: `https://data.mendeley.com/public-files/datasets/ydrm75xywg/files/61f788d6-65ce-4265-a23a-5ba16931d18b/file_downloaded` (**Stenosis detection.zip**, ~7.7 GB).
2. Kaggle → **Datasets → New Dataset** → drag the zip in. Keep **"Extract archive"** ON (Kaggle unzips server-side into read-only `/kaggle/input`).
3. Title it with the word **danilov** (e.g. `danilov-stenosis`) so cell 6 auto-detects it.
4. Back here: **Add Input → Datasets → danilov-stenosis**.

The cell below only *verifies* the attach — no download. Skip the dataset entirely for an ARCADE-only run.

In [ ]:
# Verify Danilov attach — NO download here (upload as a Kaggle Dataset named '*danilov*', same as ARCADE).
# Detects the dataset at ANY mount depth under /kaggle/input.
import glob, os
def find_danilov():
    dirs = [os.path.normpath(d) for d in glob.glob('/kaggle/input/**/', recursive=True)
            if 'danilov' in os.path.basename(os.path.normpath(d)).lower()]
    return min(dirs, key=len) if dirs else ''
_dan = find_danilov()
print('Danilov root:', _dan or 'NONE  ->  + Add Input a *danilov* dataset, or run ARCADE-only')
if _dan:
    print('  bmp:', len(glob.glob(_dan + '/**/*.bmp', recursive=True)),
          '| xml:', len(glob.glob(_dan + '/**/*.xml', recursive=True)))

In [ ]:
import glob, os
# auto-discover the ARCADE root (folder that CONTAINS stenosis/ + syntax/), wherever Kaggle nested it
hits = glob.glob('/kaggle/input/**/stenosis/*/annotations/*.json', recursive=True)
assert hits, 'No ARCADE stenosis json under /kaggle/input — attach the ARCADE dataset (Add Input).'
ARCADE_INPUT = hits[0].split('/stenosis/')[0]     # parent of stenosis/
print('inputs attached:', os.listdir('/kaggle/input'))
# Danilov: auto-pick an attached dataset whose folder name contains "danilov"
# (name your uploaded Kaggle Dataset "danilov-..."). Else hard-set DANILOV_INPUT from the list above.
# recursive: match a *danilov* folder at ANY depth (Kaggle may nest under datasets/<user>/<slug>)
dcand = [os.path.normpath(d) for d in glob.glob('/kaggle/input/**/', recursive=True)
         if 'danilov' in os.path.basename(os.path.normpath(d)).lower()]
DANILOV_INPUT = min(dcand, key=len) if dcand else ''   # '' -> ARCADE-only
print('ARCADE_INPUT  =', ARCADE_INPUT)
print('DANILOV_INPUT =', DANILOV_INPUT or '(none - ARCADE only)')
os.makedirs('data/raw', exist_ok=True)
!ln -sfn {ARCADE_INPUT} data/raw/arcade
if DANILOV_INPUT:
    !ln -sfn {DANILOV_INPUT} data/raw/danilov
n = len(glob.glob('data/raw/arcade/stenosis/**/*.json', recursive=True))
print('stenosis json via symlink:', n); assert n, 'symlink path wrong'

## 3 · Standardize ARCADE stenosis (+ Danilov) → YOLO

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/stenosis_yolo.yaml'))
if not DANILOV_INPUT:
    cfg['datasets'].pop('danilov', None)                     # ARCADE-only if no Danilov attached
from src.data_prep import danilov_to_yolo
danilov_to_yolo.main(cfg)
print('train imgs:', len(glob.glob('data/processed/stenosis/images/train/*')),
      '| val imgs:', len(glob.glob('data/processed/stenosis/images/val/*')))

## 4 · Train YOLO11s @ 768 on GPU
Set `DRY_RUN = True` for a 3-epoch wiring check; `False` for the real 150-epoch run (target **F1 > 0.57**).
Run folder auto-named via `run_tag` (e.g. `arcade+danilov_yolo11s_768_e150`). If the T4 OOMs at 768/batch16, set `train.batch: 8` in the config. Weights land in `/kaggle/working`.

In [ ]:
DRY_RUN = True
if DRY_RUN:
    cfg['train']['epochs'] = 3
    cfg['ssl'] = {'pseudo_label': False}                     # skip SSL on the fast pass
# Speed knobs (cache/workers/patience/amp) come from configs/stenosis_yolo.yaml via train_kwargs.
# Open-vocab cold start: set cfg['ssl']['seed'] = 'gdino' (needs: pip install transformers, and
# cfg['ssl']['unlabeled_dir'] pointing at the unlabeled frames) to have Grounding DINO label them
# before self-training.
from src.train.train_detector import train, train_kwargs, run_tag
TAG = run_tag(cfg)                                            # e.g. arcade_yolo11n_640_e150 -> unique per config
print('run:', TAG, '| kwargs ->', train_kwargs(cfg), '| ssl.seed:', cfg.get('ssl', {}).get('seed'))
best = train(cfg, project=f"{E['runs']}/stenosis/{TAG}")     # runs/stenosis/<TAG>/base/weights/best.pt
print('best weights ->', best)

## 5 · Export handoff
Download `best.pt` from the notebook output. On your **Mac**: `python -m src.export.yolo_to_coreml --weights best.pt` → `.mlpackage` (one Ultralytics call, NMS baked in), then `src.serve.realtime --task det`.

In [ ]:
print('Download this file from the output panel, then export CoreML on the Mac:')
print(' ', best)